In [1]:
import pandas as pd
import numpy as np

data_path = 'Ames_Housing.csv'
df = pd.read_csv(data_path)

# عرض أول 5 صفوف للتأكد من التحميل
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [2]:
# فحص عدد الصفوف والأعمدة
print(f"أبعاد البيانات: {df.shape[0]} صف و {df.shape[1]} عمود")

# عرض معلومات الأعمدة وأنواعها
df.info()

أبعاد البيانات: 2930 صف و 82 عمود
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS SubClass      2930 non-null   int64  
 3   MS Zoning        2930 non-null   object 
 4   Lot Frontage     2440 non-null   float64
 5   Lot Area         2930 non-null   int64  
 6   Street           2930 non-null   object 
 7   Alley            198 non-null    object 
 8   Lot Shape        2930 non-null   object 
 9   Land Contour     2930 non-null   object 
 10  Utilities        2930 non-null   object 
 11  Lot Config       2930 non-null   object 
 12  Land Slope       2930 non-null   object 
 13  Neighborhood     2930 non-null   object 
 14  Condition 1      2930 non-null   object 
 15  Condition 2      2930 non-null   object 
 16  Bldg Type        2930 non-

In [3]:
# حساب عدد القيم المفقودة في كل عمود وترتيبها تنازلياً
missing_values = df.isnull().sum()
print("الأعمدة التي تحتوي على قيم مفقودة:")
print(missing_values[missing_values > 0].sort_values(ascending=False))

الأعمدة التي تحتوي على قيم مفقودة:
Pool QC           2917
Misc Feature      2824
Alley             2732
Fence             2358
Mas Vnr Type      1775
Fireplace Qu      1422
Lot Frontage       490
Garage Qual        159
Garage Cond        159
Garage Yr Blt      159
Garage Finish      159
Garage Type        157
Bsmt Exposure       83
BsmtFin Type 2      81
Bsmt Cond           80
Bsmt Qual           80
BsmtFin Type 1      80
Mas Vnr Area        23
Bsmt Full Bath       2
Bsmt Half Bath       2
BsmtFin SF 1         1
BsmtFin SF 2         1
Electrical           1
Total Bsmt SF        1
Bsmt Unf SF          1
Garage Area          1
Garage Cars          1
dtype: int64


In [5]:
def clean_data(data_frame):
    # أول شي، ناخذ نسخة من البيانات عشان ما نحوس الجدول الأصلي ونقدر نرجع له لو احتجنا [cite: 31, 38]
    df_cleaned = data_frame.copy()

    # 1. نشيل أي سطور مكررة مالها داعي عشان تكون بياناتنا دقيقة [cite: 39, 66]
    df_cleaned.drop_duplicates(inplace=True)

    # 2. تنظيف "الحوسة" والقيم الفاضية (Missing Values) [cite: 39, 56]
    # هنا حذفت الأعمدة اللي أغلبها فاضي (زي المسبح والممر) لأنها بتشوش على التحليل [cite: 47, 65]
    cols_to_drop = ['Pool QC', 'Misc Feature', 'Alley', 'Fence']
    df_cleaned.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # الأعمدة اللي فيها أرقام وفاضية، بنعبيها بـ "الوسيط" (Median) لأنه أضمن وما يتأثر بالقيم الشاطحة [cite: 39, 42]
    numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

    # الأعمدة اللي فيها نصوص، بنعبيها بأكثر قيمة متكررة (Mode) عشان نمشي الأمور [cite: 39, 64]
    cat_cols = df_cleaned.select_dtypes(include=['object']).columns
    for col in cat_cols:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])

    # 3. ترويض القيم الشاطحة (Outliers) في عمود السعر [cite: 39, 67]
    # قصينا الأسعار اللي فوق النسبة 99% عشان ما تخرب علينا الحسابات بعدين [cite: 68, 107]
    upper_limit = df_cleaned['SalePrice'].quantile(0.99)
    df_cleaned['SalePrice'] = np.where(df_cleaned['SalePrice'] > upper_limit, upper_limit, df_cleaned['SalePrice'])

    return df_cleaned

# نطبق الدالة اللي تعبنا فيها على بياناتنا ونشوف النتيجة [cite: 69, 121]
df_final = clean_data(df)

# خلونا نتأكد إن كل شي تمام قبل ما ننتقل للي بعده (Checks) [cite: 70, 129]
print("--- فحص سريع للبيانات بعد التنظيف ---")
print(f"1. هل لسى فيه قيم فاضية؟ {df_final.isnull().any().any()} (إذا False يعني شغلنا صح)")
print(f"2. هل الأسعار منطقية وأكبر من صفر؟ {(df_final['SalePrice'] > 0).all()}")
print(f"3. كم عمود صفينا عليه في الأخير؟ {df_final.shape[1]}")

--- فحص سريع للبيانات بعد التنظيف ---
1. هل لسى فيه قيم فاضية؟ False (إذا False يعني شغلنا صح)
2. هل الأسعار منطقية وأكبر من صفر؟ True
3. كم عمود صفينا عليه في الأخير؟ 78


In [6]:
# حفظ البيانات المنظفة في ملف جديد
df_final.to_csv('Ames_Housing_Cleaned.csv', index=False)

تم حفظ الملف بنجاح! جاهزين للمرحلة الثانية.
